<a href="https://colab.research.google.com/github/danish2k04/Getting-into-Pytorch-Deep-Learning/blob/main/efficient_net_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install pandas scikit-learn

import os, random, numpy as np, pandas as pd, torch, torch.nn as nn
from PIL import Image, ImageOps
from tqdm.auto import tqdm
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from torchvision import models, transforms


In [2]:
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

from google.colab import drive
drive.mount('/content/drive')

ROOT       = "/content/drive/MyDrive/pap_cell_project/data"
TRAIN_CSV  = os.path.join(ROOT, "isbi2025-ps3c-train-dataset.csv")

Using device: cuda
Mounted at /content/drive


In [3]:
!apt-get install -y p7zip-full -q

Reading package lists...
Building dependency tree...
Reading state information...
p7zip-full is already the newest version (16.02+dfsg-8).
0 upgraded, 0 newly installed, 0 to remove and 53 not upgraded.


In [4]:
DATA_DIR = "/content/apacc_data"
os.makedirs(DATA_DIR, exist_ok=True)
!7z x "{os.path.join(ROOT, 'isbi2025-ps3c-train-dataset.7z')}" -o{DATA_DIR} -y


7-Zip [64] 16.02 : Copyright (c) 1999-2016 Igor Pavlov : 2016-05-21
p7zip Version 16.02 (locale=en_US.UTF-8,Utf16=on,HugeFiles=on,64 bits,2 CPUs Intel(R) Xeon(R) CPU @ 2.00GHz (50653),ASM,AES-NI)

Scanning the drive for archives:
  0M Scan /content/drive/MyDrive/pap_cell_project/data/                                                       1 file, 16087644759 bytes (15 GiB)

Extracting archive: /content/drive/MyDrive/pap_cell_project/data/isbi2025-ps3c-train-dataset.7z
--
Path = /content/drive/MyDrive/pap_cell_project/data/isbi2025-ps3c-train-dataset.7z
Type = 7z
Physical Size = 16087644759
Headers Size = 1272355
Method = LZMA2:26
Solid = +
Blocks = 1

  0%      0% 66 - unhealthy/isbi2025_ps3c_train_image_82781.png                                                       

In [5]:
DROP_BOTHCELLS = True
IMG_SIZE       = 300          # EfficientNetV2-S native resolution
BATCH_SIZE     = 32           # reduced from 64 to fit larger images in GPU memory
EPOCHS         = 35
CHECKPOINT_PATH = os.path.join(ROOT, "checkpoint.pth")



In [6]:
class PadToSquareWhite:
    def __call__(self, img):
        w, h = img.size
        m = max(w, h)
        l = (m - w) // 2
        t = (m - h) // 2
        return ImageOps.expand(img, border=(l, t, m - w - l, m - h - t), fill=(255, 255, 255))

train_tf = transforms.Compose([
    PadToSquareWhite(),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20, fill=255),
    transforms.ColorJitter(brightness=0.08, contrast=0.08, saturation=0.08),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])

eval_tf = transforms.Compose([
    PadToSquareWhite(),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])


In [7]:
df = pd.read_csv(TRAIN_CSV)
if DROP_BOTHCELLS:
    df = df[df["label"] != "bothcells"].copy()

# Classes are exactly: healthy, rubbish, unhealthy
classes      = sorted(df["label"].unique())
class_to_idx = {c: i for i, c in enumerate(classes)}
df["target"] = df["label"].map(class_to_idx)

print("Classes:", classes)
print("Class → index:", class_to_idx)
print("Label distribution:\n", df["label"].value_counts())

Classes: ['healthy', 'rubbish', 'unhealthy']
Class → index: {'healthy': 0, 'rubbish': 1, 'unhealthy': 2}
Label distribution:
 label
rubbish      50371
healthy      28895
unhealthy     2366
Name: count, dtype: int64


In [8]:
class PapCellDataset(Dataset):
    def __init__(self, df, img_root, tfm):
        self.df       = df.reset_index(drop=True)
        self.img_root = img_root
        self.tfm      = tfm

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        img_path = os.path.join(self.img_root, row["label"], row["image_name"])
        img      = Image.open(img_path).convert("RGB")
        return self.tfm(img), int(row["target"])

In [9]:
TRAIN_DIR = DATA_DIR

sss = StratifiedShuffleSplit(n_splits=1, test_size=0.15, random_state=SEED)
train_idx, val_idx = next(sss.split(df["image_name"], df["target"]))
df_train = df.iloc[train_idx].copy()
df_val   = df.iloc[val_idx].copy()

train_ds = PapCellDataset(df_train, TRAIN_DIR, train_tf)
val_ds   = PapCellDataset(df_val,   TRAIN_DIR, eval_tf)

print(f"Train samples: {len(train_ds)} | Val samples: {len(val_ds)}")


Train samples: 69387 | Val samples: 12245


In [10]:
counts        = df_train["target"].value_counts().sort_index().values.astype(np.float32)
class_weights = 1.0 / counts                          # full inverse — not sqrt
class_weights = class_weights / class_weights.mean()  # normalise

sample_weights = df_train["target"].map(
    {i: w for i, w in enumerate(class_weights)}
).values

sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=2, pin_memory=True, persistent_workers=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True, persistent_workers=True)

print("Class weights:", dict(zip(classes, class_weights)))

Class weights: {'healthy': np.float32(0.2175966), 'rubbish': np.float32(0.12482518), 'unhealthy': np.float32(2.6575782)}


In [11]:
model = models.efficientnet_v2_s(weights=models.EfficientNet_V2_S_Weights.DEFAULT)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, len(classes))
model = model.to(device)

criterion = nn.CrossEntropyLoss(
    weight=torch.tensor(class_weights, dtype=torch.float32, device=device),
    label_smoothing=0.05
)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler    = torch.amp.GradScaler(enabled=(device == "cuda"))

best_state, best_macro_f1 = None, -1.0
print("Model ready:", model.__class__.__name__)


Downloading: "https://download.pytorch.org/models/efficientnet_v2_s-dd5fe13b.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_v2_s-dd5fe13b.pth


100%|██████████| 82.7M/82.7M [00:00<00:00, 145MB/s]


Model ready: EfficientNet


In [12]:
def run_epoch(loader, train):
    model.train(train)
    all_y, all_p, total_loss = [], [], 0.0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for x, y in tqdm(loader, desc="train" if train else "val ", leave=False):
            x, y = x.to(device), y.to(device)
            if train:
                optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type="cuda", enabled=(device == "cuda")):
                logits = model(x)
                loss   = criterion(logits, y)
            if train:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            total_loss += loss.item() * x.size(0)
            all_y.append(y.cpu().numpy())
            all_p.append(logits.argmax(1).cpu().numpy())

    y_true = np.concatenate(all_y)
    y_pred = np.concatenate(all_p)
    return (
        total_loss / len(y_true),
        accuracy_score(y_true, y_pred),
        f1_score(y_true, y_pred, average="macro"),
        f1_score(y_true, y_pred, average="weighted"),
        y_true,
        y_pred,
    )

In [ ]:
start_epoch     = 1
patience        = 6
patience_counter = 0

if os.path.exists(CHECKPOINT_PATH):
    print(f"Checkpoint found — resuming from {CHECKPOINT_PATH}")
    ckpt = torch.load(CHECKPOINT_PATH, map_location=device)
    model.load_state_dict(ckpt["model_state"])
    optimizer.load_state_dict(ckpt["optimizer_state"])
    scheduler.load_state_dict(ckpt["scheduler_state"])
    scaler.load_state_dict(ckpt["scaler_state"])
    best_state      = ckpt["best_state"]
    best_macro_f1   = ckpt["best_macro_f1"]
    start_epoch     = ckpt["epoch"] + 1
    patience_counter = ckpt["patience_counter"]
    print(f"Resuming from epoch {start_epoch} | best macro-F1 so far: {best_macro_f1:.4f}")
else:
    print("No checkpoint found — starting fresh.")

for epoch in range(start_epoch, EPOCHS + 1):
    tr = run_epoch(train_loader, train=True)
    va = run_epoch(val_loader,   train=False)
    scheduler.step()

    print(f"Epoch {epoch:02d} | "
          f"train_loss={tr[0]:.4f} | val_loss={va[0]:.4f} | "
          f"val_acc={va[1]:.4f} | val_macro_f1={va[2]:.4f} | val_weighted_f1={va[3]:.4f}")

    if va[2] > best_macro_f1:
        best_macro_f1    = va[2]
        best_state       = {k: v.cpu() for k, v in model.state_dict().items()}
        patience_counter = 0
        # also save best model separately so it survives even if checkpoint is overwritten
        torch.save(best_state, os.path.join(ROOT, "best_model.pth"))
        print(f"  ✓ New best macro-F1: {best_macro_f1:.4f} — saved best_model.pth")
    else:
        patience_counter += 1

    # save full checkpoint every epoch — survives Colab disconnect
    torch.save({
        "epoch":           epoch,
        "model_state":     model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "scheduler_state": scheduler.state_dict(),
        "scaler_state":    scaler.state_dict(),
        "best_state":      best_state,
        "best_macro_f1":   best_macro_f1,
        "patience_counter": patience_counter,
    }, CHECKPOINT_PATH)

    if patience_counter >= patience:
        print(f"Early stopping at epoch {epoch} — no macro-F1 improvement for {patience} epochs.")
        break

print(f"\nTraining done. Best val macro-F1: {best_macro_f1:.4f}")

In [13]:
model = models.efficientnet_v2_s(weights=models.EfficientNet_V2_S_Weights.DEFAULT)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, len(classes))
model = model.to(device)

model.load_state_dict(torch.load(os.path.join(ROOT, "best_model.pth"), map_location=device))
model.eval()
print("Best model loaded successfully.")

Best model loaded successfully.


In [14]:
TEST_DATA_DIR = "/content/apacc_test_data"
os.makedirs(TEST_DATA_DIR, exist_ok=True)
!7z x "{os.path.join(ROOT, 'isbi2025-ps3c-test-dataset.7z')}" -o{TEST_DATA_DIR} -y

print("\nContents of TEST_DATA_DIR:")
print(os.listdir(TEST_DATA_DIR)[:10])



7-Zip [64] 16.02 : Copyright (c) 1999-2016 Igor Pavlov : 2016-05-21
p7zip Version 16.02 (locale=en_US.UTF-8,Utf16=on,HugeFiles=on,64 bits,2 CPUs Intel(R) Xeon(R) CPU @ 2.00GHz (50653),ASM,AES-NI)

Scanning the drive for archives:
  0M Scan /content/drive/MyDrive/pap_cell_project/data/                                                       1 file, 3213844601 bytes (3065 MiB)

Extracting archive: /content/drive/MyDrive/pap_cell_project/data/isbi2025-ps3c-test-dataset.7z
--
Path = /content/drive/MyDrive/pap_cell_project/data/isbi2025-ps3c-test-dataset.7z
Type = 7z
Physical Size = 3213844601
Headers Size = 275513
Method = LZMA2:26
Solid = +
Blocks = 1

  0%      1% 321 - isbi2025_ps3c_test_image_00332.png                                               2% 480 - isbi2025_ps3c_test_image_0

In [15]:
# The annotated test CSV — check which file exists in ROOT
test_csv_candidates = [
    os.path.join(ROOT, "isbi2025-ps3c-test-dataset.csv"),
    os.path.join(ROOT, "isbi2025-ps3c-test-dataset-annotated.csv"),
    "isbi2025-ps3c-test-dataset-annotated.csv",   # local fallback
]

TEST_CSV_PATH = None
for p in test_csv_candidates:
    if os.path.exists(p):
        TEST_CSV_PATH = p
        print(f"Found test CSV: {p}")
        break

if TEST_CSV_PATH is None:
    raise FileNotFoundError(
        "No test CSV found. Upload isbi2025-ps3c-test-dataset.csv to your Drive ROOT folder."
    )

df_test = pd.read_csv(TEST_CSV_PATH)
if DROP_BOTHCELLS:
    df_test = df_test[df_test["label"] != "bothcells"].copy()
df_test["target"] = df_test["label"].map(class_to_idx)

# Sanity check — must be zero overlap with val split
overlap = set(df_test["image_name"]).intersection(set(df_val["image_name"]))
print(f"Overlap between test and val sets: {len(overlap)} (must be 0)")
print(f"Test samples: {len(df_test)}")
print(df_test["label"].value_counts())

Found test CSV: /content/drive/MyDrive/pap_cell_project/data/isbi2025-ps3c-test-dataset-annotated.csv
Overlap between test and val sets: 0 (must be 0)
Test samples: 18159
label
rubbish      11757
healthy       5826
unhealthy      576
Name: count, dtype: int64


In [16]:
# Test images are flat in TEST_DATA_DIR (no subfolders by label)
# so we need a slightly different dataset class for test
class PapCellTestDataset(Dataset):
    def __init__(self, df, img_root, tfm):
        self.df       = df.reset_index(drop=True)
        self.img_root = img_root
        self.tfm      = tfm

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        img_path = os.path.join(self.img_root, row["image_name"])
        img      = Image.open(img_path).convert("RGB")
        return self.tfm(img), int(row["target"])

test_ds     = PapCellTestDataset(df_test, TEST_DATA_DIR, eval_tf)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=2, pin_memory=True)

print(f"Test samples loaded: {len(test_ds)}")

Test samples loaded: 18159


In [17]:
counts        = df["target"].value_counts().sort_index().values.astype(np.float32)
class_weights = 1.0 / counts
class_weights = class_weights / class_weights.mean()

criterion = nn.CrossEntropyLoss(
    weight=torch.tensor(class_weights, dtype=torch.float32, device=device),
    label_smoothing=0.05
)
print("Criterion ready.")

Criterion ready.


In [18]:
model.eval()
te = run_epoch(test_loader, train=False)

print("\nTest Report (healthy / rubbish / unhealthy only):")
print(classification_report(te[4], te[5], target_names=classes, digits=4))
print("Confusion Matrix:\n", confusion_matrix(te[4], te[5]))
print(f"\nTest macro-F1:    {te[2]:.4f}")
print(f"Test accuracy:    {te[1]:.4f}")
print(f"Test weighted-F1: {te[3]:.4f}")

val :   0%|          | 0/568 [00:00<?, ?it/s]


Test Report (healthy / rubbish / unhealthy only):
              precision    recall  f1-score   support

     healthy     0.7592    0.8469    0.8006      5826
     rubbish     0.9187    0.8593    0.8880     11757
   unhealthy     0.3665    0.4219    0.3923       576

    accuracy                         0.8415     18159
   macro avg     0.6815    0.7094    0.6936     18159
weighted avg     0.8500    0.8415    0.8443     18159

Confusion Matrix:
 [[ 4934   737   155]
 [ 1389 10103   265]
 [  176   157   243]]

Test macro-F1:    0.6936
Test accuracy:    0.8415
Test weighted-F1: 0.8443
